# Lab 49: Graded gold

[Lab 47](../47-trustworthy-gold/) kept labels binary. Move the rubric to a 0-3 ordinal scale, adjudicate the items experts split on with a senior adjudicator (not a silent majority), grade the judge with quadratic-weighted κ, and re-derive the [Lab 45](../45-anchoring-the-consensus/) annotator weights against graded gold - where the ranking flips. Fill in the `TODO` cells; reference in `solution/`.

## Step 0: Setup (inline ordinal measures)

In [ ]:
from itertools import combinations


def krippendorff_alpha_ordinal(ratings_by_item, levels):
    """Ordinal Krippendorff alpha for small teaching examples."""
    level_count = len(levels)
    index = {value: i for i, value in enumerate(levels)}
    observed = [[0.0] * level_count for _ in range(level_count)]

    for item in ratings_by_item:
        m = len(item)
        if m < 2:
            continue

        for a, b in combinations(range(m), 2):
            ca = index[item[a]]
            cb = index[item[b]]
            observed[ca][cb] += 1 / (m - 1)
            observed[cb][ca] += 1 / (m - 1)

    category_totals = [sum(observed[c]) for c in range(level_count)]
    total = sum(category_totals)

    if total == 0:
        return float("nan")

    def delta(c, k):
        lo, hi = (c, k) if c <= k else (k, c)
        span = sum(category_totals[g] for g in range(lo, hi + 1))
        span -= (category_totals[c] + category_totals[k]) / 2
        return span * span

    observed_disagreement = (
        sum(
            observed[c][k] * delta(c, k)
            for c in range(level_count)
            for k in range(level_count)
        )
        / total
    )

    expected_disagreement = (
        sum(
            category_totals[c] * category_totals[k] * delta(c, k)
            for c in range(level_count)
            for k in range(level_count)
        )
        / (total * (total - 1))
    )

    return 1 - observed_disagreement / expected_disagreement


def quadratic_weighted_kappa(y1, y2, levels):
    level_count = len(levels)
    index = {value: i for i, value in enumerate(levels)}
    observed = [[0] * level_count for _ in range(level_count)]

    for a, b in zip(y1, y2, strict=False):
        observed[index[a]][index[b]] += 1

    n_items = len(y1)
    row_totals = [sum(observed[i]) for i in range(level_count)]
    col_totals = [
        sum(observed[i][j] for i in range(level_count))
        for j in range(level_count)
    ]

    weights = [
        [((i - j) ** 2) / ((level_count - 1) ** 2) for j in range(level_count)]
        for i in range(level_count)
    ]

    numerator = sum(
        weights[i][j] * observed[i][j]
        for i in range(level_count)
        for j in range(level_count)
    )

    denominator = sum(
        weights[i][j] * row_totals[i] * col_totals[j] / n_items
        for i in range(level_count)
        for j in range(level_count)
    )

    return 1 - numerator / denominator


## Step 1: The graded set

In [ ]:
import json

# Lab 47 used binary labels. A faithfulness judgment is really ordinal: 3 fully supported,
# 2 a minor unsupported detail, 1 partially, 0 unsupported/contradicted. Load the graded set:
# three annotators, three experts, a senior adjudicator label on the items experts split, and
# the judge - all on 0-3.
LEVELS=[0,1,2,3]
with open("./graded_gold.jsonl") as f:
    items = [json.loads(line) for line in f]
def col(k):
    return [it[k] for it in items]
def med3(a, b, c):
    return sorted([a, b, c])[1]
def is_wide(it):
    expert_scores = [it["e1"], it["e2"], it["e3"]]
    return max(expert_scores) - min(expert_scores) >= 2
consensus   = [med3(it["a1"],it["a2"],it["a3"]) for it in items]
gold_median = [med3(it["e1"],it["e2"],it["e3"]) for it in items]              # majority-of-experts
gold_senior = [it.get("senior", gold_median[i]) for i,it in enumerate(items)] # senior on splits
print(f"{len(items)} items on 0-3; experts split on {sum(is_wide(it) for it in items)} items")

## Step 2: Are experts a tighter anchor, graded?

In [ ]:
# TODO: compute krippendorff_alpha_ordinal over the three annotators and over the three
# experts (lists of [r1,r2,r3] per item, LEVELS=[0,1,2,3]). Confirm experts > annotators.
raise NotImplementedError

## Step 3: Majority-of-experts vs a senior adjudicator

In [ ]:
# Two ways to build gold from the experts. Where experts split widely, a majority/median
# silently picks the middle; a senior adjudicator makes a reasoned call with the guideline.
split=[it["id"] for it in items if is_wide(it)]
print("expert split (adjudication queue):", split)
for i,it in enumerate(items):
    if is_wide(it):
        print(f"  {it['id']}: experts={[it['e1'],it['e2'],it['e3']]} median={gold_median[i]} senior={gold_senior[i]}")
diff=[items[i]["id"] for i in range(len(items)) if gold_median[i]!=gold_senior[i]]
print(f"the two protocols disagree on {diff} - exactly the items worth a human decision")

## Step 4: Grade the judge

In [ ]:
# Grade the judge. Binary (pass/fail at a threshold) collapses a 2-vs-3 near-miss and a
# 0-vs-3 blunder into the same "wrong". The ordinal view separates them.
jg = quadratic_weighted_kappa(col("judge"), gold_senior, LEVELS)
jc = quadratic_weighted_kappa(col("judge"), consensus,  LEVELS)
mae = sum(
    abs(j - g)
    for j, g in zip(col("judge"), gold_senior, strict=False)
) / len(items)
off1 = sum(
    1
    for j, g in zip(col("judge"), gold_senior, strict=False)
    if abs(j - g) == 1
) / len(items)
print(f"judge vs gold:      QWK {jg:.2f}, MAE {mae:.2f}, off-by-one {off1:.0%}")
print(f"judge vs consensus: QWK {jc:.2f}")
print("The judge is systematically one level low on high-faithfulness answers - a near-miss")
print("bias the binary gate would either hide or punish as a full miss. Graded, you can")
print("calibrate it (shift the threshold) instead of discarding the judge.")

## Step 5: Re-derive the Lab 45 weights against gold

In [ ]:
# TODO: with quadratic_weighted_kappa, weight each annotator vs the consensus and vs
# gold_senior; rank both ways. Confirm the ranking flips - a1 is last vs the biased consensus
# but first vs gold.
raise NotImplementedError

## Step 6: The same move as the operations side

In [ ]:
# Same move as the operations side (Lab 48): make the thing real instead of a stand-in.
# There it was a file lock -> Redis, claim-before-send -> dead-letter, raw bytes -> normalized.
# Here it is binary -> ordinal, a silent majority -> a senior adjudicator on the split items,
# and weights anchored to a biased consensus -> weights re-derived against graded gold.
print("Graded rubric, real adjudication, weights re-derived against gold - the evaluation")
print("anchor is now as production-shaped as the operations backend.")

## Step 7: The discipline

In [ ]:
# The discipline: a binary label throws away the distance between answers, and a majority
# vote hides the disagreements that most need a human. Grade the rubric, adjudicate the
# splits, and re-derive every weight and ceiling against the graded gold - not the consensus.
print("Score the distance, adjudicate the splits, re-anchor the weights. Binary and majority")
print("were the convenient approximations; graded and adjudicated are the real ones.")

## What you built

The graded, adjudicated version of the evaluation anchor: an ordinal 0-3 faithfulness rubric scored with the right tools (Krippendorff's α with the ordinal metric for multi-rater agreement, quadratic-weighted κ for pairwise graded agreement); inter-expert α above inter-annotator α even on the graded scale; a senior-adjudicator protocol that resolves the wide expert splits differently from a silent majority/median; a graded judge evaluation (QWK, MAE, off-by-one) that exposes a systematic one-level-low bias the binary gate would hide; and the Lab 45 annotator weights re-derived against graded gold, where the ranking flips - the annotator that looked worst against the biased consensus is best against gold.

**Where this simplifies:** three experts / 24 items / a single faithfulness dimension is a teaching size (real rubrics are multi-dimensional - faithfulness × relevance × completeness - and larger); the ordinal metric treats the 0-3 steps as evenly spaced, which a real rubric should justify; the senior adjudicator is a single stand-in for a documented protocol (a second senior, a tie-break rule, a written rationale); and graded gold is still an anchor, not truth - report its inter-expert α alongside it.

This is the graded close of the evaluation thread: [Lab 40](../40-annotation-quality/) set a ceiling, [Lab 45](../45-anchoring-the-consensus/) anchored the consensus, [Lab 47](../47-trustworthy-gold/) made gold multi-expert, and this makes it graded and adjudicated - then re-derives the weights against it.